In [11]:
import re
import unicodedata


def find_if_incrementing_or_repeating(splits, test_repeating=False):
    """Finds if the given list of words is incrementing
    
    A sequence is incrementing if it there are a set of integers or decimals in arithmetic progression 
    intersperced with or without repeating characters
    
    Args:
        splits (list): List of values to be analyzed.

    Returns:
        Tuple(bool, int): A tuple containing a boolean value indicating if 
            the sequence is incrementing and the calculated difference.
    """
    # We need atleast 3 integers to define an AP
    if len(splits) < 3 and not test_repeating:
        return False, 0
    elif len(splits) == 3 and not test_repeating:
        # Every element has to be a number
        if not all([type(i) in [float, int] for i in splits]): return False, 0
        return (splits[2] - 2*splits[1] + splits[0]) < 1e-5, splits[2] - splits[1]
    
    # First and last words of a sequence can be partial
    # We ignore them if length of splits is more than 4
    if len(splits) > 4 and not test_repeating:
        splits = splits[1:-1]
        
        
    is_num_inc = False
    diff = None
    
    for temp_len in range(1, len(splits)//2 + 1):
        is_inc = True
        diff_valid = False
        for every_i in range(temp_len):
            curr_diff = None
            for i in range(every_i + temp_len, len(splits), temp_len):            
                    
                if type(splits[i]) != type(splits[i - temp_len]):
                    is_inc = False
                    break
                
                if curr_diff is None and type(splits[i]) in [int, float]:
                    curr_diff = splits[i] - splits[i - temp_len]
                
                elif type(splits[i]) in [int, float]:
                    is_curr_inc = abs(splits[i] - splits[i - temp_len] - curr_diff) < 1e-10       
                    if not is_curr_inc:
                        is_inc = False
                        break
                    else:
                        diff_valid = True
                elif type(splits[i]) == str and splits[i] != splits[i - temp_len]:
                    is_inc = False
                    break
            
            if not is_inc:
                break
            
            if diff is None and curr_diff is not None and curr_diff != 0:
                diff = curr_diff
        if is_inc:
            if diff is not None and diff_valid:
                return True, diff
            elif diff is None:
                return True, None
    
    return False, 0

def split_text(text, split_type = "incrementing"):

    if split_type == "repeating":
        return list(text)

    elif split_type != "incrementing":
        raise ValueError("Invalid Split Type")

    # Check if we have hexadecimal numerals
    text = re.sub(r"\s+", " ", text)
    splits = []
    text_recon = ""
    for word in text.split(" "):
        try:
            text_recon += str(int(word, 0)) + " "
        except ValueError:
            text_recon += word + " "
    
    text_recon = repr(text_recon.strip())
    for word in text_recon.split(" "):
        
        
        # Replace escape characters
        word = word.replace("\\n", "")
        word = word.replace("\\x", "")
        word = word.replace("\\", "")
        word = word.replace("\'", "")
        word = word.replace("\"", "")
        
        
        word = re.split("([0-9]+)", word)
        splits.extend(word)
        
        
    splits_new = []
    to_continue = False
    for idx, word in enumerate(splits):
        word = word.strip("\'")
        if to_continue:
            to_continue = not to_continue
            continue
        if word.strip(" ") == "":
            continue
        
        if word == "":
            continue
        
        try:
            splits_new.append(int(word))
        except ValueError:
            splits_new.append(word)
    
    return splits_new
        
def is_pattern(text):
    splits = split_text(text)
    is_inc, diff = find_if_incrementing_or_repeating(splits)
    if is_inc and diff is not None and diff != 0:
        return True, False
    
    splits = split_text(text, split_type="repeating") 
    is_inc, diff = find_if_incrementing_or_repeating(splits)
    if is_inc and diff is not None and diff != 0:
        return True, False
    elif is_inc:
        return False, True
    else:
        return False, False

In [13]:
is_pattern("123123213123213123123123123123123")

[123123213123213123123123123123123]
['1', '2', '3', '1', '2', '3', '2', '1', '3', '1', '2', '3', '2', '1', '3', '1', '2', '3', '1', '2', '3', '1', '2', '3', '1', '2', '3', '1', '2', '3', '1', '2', '3']


(False, False)

In [1]:
DS_PATH = "/mnt/ssd-1/sai/semantic-memorization/datasets/2024-05-30_19-50-03/pile_deduped_12b/part-00000-e0328024-f04b-44c1-987c-33c851ba9a38-c000.snappy.parquet"

In [2]:
import pandas as pd

In [3]:
data = pd.read_parquet(DS_PATH)

In [12]:
data[data['is_repeating'] == True].head(10)['text'].iloc[4]

'01 9/24/01 9/24/01 9/24/01 9/24/01 9/24/01 9/24/01 9/24/01 9/24/01 9/24/01 9/24/01 9/24/01 9/24/01 9/24'

In [1]:
from datasets import load_dataset

In [2]:
ds = load_dataset("usvsnsp/generation-semantic-filters")

In [4]:
ds = ds.topandas()

AttributeError: 'DatasetDict' object has no attribute 'topandas'

In [5]:
ds

DatasetDict({
    pile_duped_6.9b: Dataset({
        features: ['sequence_id', 'tokens', 'text', 'is_incrementing', 'is_repeating', 'sequence_duplicates', 'max_frequency', 'avg_frequency', 'min_frequency', 'median_frequency', 'p25_frequency', 'p75_frequency', 'frequencies', 'nl_scores', '0_8_snowclones', '0_9_snowclones', '0_8_templates', '0_9_templates', 'huffman_coding_length', 'memorization_score', 'index', 'loss', 'prompt_perplexity', 'generation_perplexity', 'sequence_perplexity'],
        num_rows: 5000000
    })
    memories_duped_1b: Dataset({
        features: ['sequence_id', 'tokens', 'text', 'is_incrementing', 'is_repeating', 'sequence_duplicates', 'max_frequency', 'avg_frequency', 'min_frequency', 'median_frequency', 'p25_frequency', 'p75_frequency', 'frequencies', 'nl_scores', '0_8_snowclones', '0_9_snowclones', '0_8_templates', '0_9_templates', 'huffman_coding_length', 'memorization_score', 'index', 'loss', 'prompt_perplexity', 'generation_perplexity', 'sequence_perplexit

In [12]:
PATHS = ['/mnt/ssd-1/sai/semantic-memorization/datasets/2024-05-30_19-50-03', '/mnt/ssd-1/sai/semantic-memorization/datasets/2024-06-01_18-59-46']

In [17]:
import pandas as pd

In [19]:
datasets = {}
import os
for path in PATHS:
    for dataset in tqdm(os.listdir(path)):
        if not os.path.isdir(os.path.join(path, dataset)):
            continue
        for file in os.listdir(os.path.join(path, dataset)):
            file_path = os.path.join(path, dataset, file)            
            if file.endswith(".parquet"):
                datasets[dataset] = pd.read_parquet(file_path)

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

In [22]:
datasets.keys()

dict_keys(['pile_duped_160m', 'memories_deduped_70m', 'memories_duped_70m', 'pile_deduped_160m', 'pile_duped_12b', 'pile_deduped_1b', 'pile_duped_410m', 'pile_deduped_410m', 'memories_duped_1b', 'pile_duped_1b', 'memories_duped_6.9b', 'pile_duped_1.4b', 'memories_deduped_160m', 'memories_deduped_410m', 'pile_deduped_2.8b', 'pile_deduped_12b', 'pile_duped_6.9b', 'pile_duped_2.8b', 'memories_deduped_6.9b', 'memories_deduped_1b', 'memories_duped_410m', 'pile_duped_70m', 'memories_duped_1.4b', 'pile_deduped_70m', 'memories_deduped_2.8b', 'pile_deduped_6.9b', 'memories_duped_2.8b', 'pile_deduped_1.4b', 'memories_deduped_12b', 'memories_duped_160m', 'memories_deduped_1.4b', 'memories_duped_12b'])

In [21]:
from tqdm.auto import tqdm
from datasets import Dataset

In [ ]:
for key in tqdm(ds):
    ds_key = ds[key]
    data = ds_key.to_pandas()[['sequence_id', 'loss', 'prompt_perplexity', 'generation_perplexity', 'sequence_perplexity']]
    pd_ds = datasets[key].set_index("sequence_id")
    data = data.join(pd_ds, on="sequence_id")
    data = Dataset.from_pandas(data)
    data.push_to_hub("usvsnsp/semantic-filters", split=key, max_shard_size="10GB")

  0%|          | 0/32 [00:00<?, ?it/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/5000 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1257 [00:00<?, ?ba/s]

...

Creating parquet from Arrow format:   0%|          | 0/582 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/5.37k [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/5000 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/5.52k [00:00<?, ?B/s]

In [7]:
from datasets import load_dataset
from tqdm.auto import tqdm
import pandas as pd

In [2]:
final_ds = load_dataset("usvsnsp/semantic-filters")

In [3]:
PATHS = ['/mnt/ssd-1/sai/semantic-memorization/datasets/2024-06-04_11-46-03', '/mnt/ssd-1/sai/semantic-memorization/datasets/2024-06-04_18-29-08', '/mnt/ssd-1/sai/semantic-memorization/datasets/2024-06-03_11-56-07']

In [8]:
datasets = {}
import os
for path in PATHS:
    for dataset in tqdm(os.listdir(path)):
        if not os.path.isdir(os.path.join(path, dataset)):
            continue
        for file in os.listdir(os.path.join(path, dataset)):
            file_path = os.path.join(path, dataset, file)            
            if file.endswith(".parquet"):
                datasets[dataset] = pd.read_parquet(file_path)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/11 [00:00<?, ?it/s]

In [10]:
datasets.keys()

dict_keys(['memories_duped_12b.103000', 'memories_duped_12b.123000', 'memories_deduped_12b.103000', 'memories_duped_12b.63000', 'memories_deduped_12b.63000', 'memories_deduped_12b.23000', 'memories_deduped_12b.123000', 'memories_duped_12b.83000', 'memories_duped_12b.23000', 'memories_deduped_12b.83000', 'memories_duped_12b.43000', 'memories_deduped_12b.43000'])

In [12]:
len(datasets.keys())

12

In [13]:
datasets['memories_duped_12b.103000'].columns

Index(['sequence_id', 'text', 'is_incrementing', 'is_repeating',
       'sequence_duplicates', 'max_frequency', 'avg_frequency',
       'min_frequency', 'median_frequency', 'p25_frequency', 'p75_frequency',
       'frequencies', 'tokens', 'repeating_offset', 'num_repeating',
       'smallest_repeating_chunk', 'nl_scores', 'huffman_coding_length',
       'memorization_score'],
      dtype='object')

In [14]:
final_ds['memories_duped_12b']

Dataset({
    features: ['sequence_id', 'loss', 'prompt_perplexity', 'generation_perplexity', 'sequence_perplexity', 'text', 'is_incrementing', 'is_repeating', 'sequence_duplicates', 'max_frequency', 'avg_frequency', 'min_frequency', 'median_frequency', 'p25_frequency', 'p75_frequency', 'frequencies', 'tokens', 'repeating_offset', 'num_repeating', 'smallest_repeating_chunk', 'nl_scores', '0_8_snowclones', '0_9_snowclones', '0_8_templates', '0_9_templates', 'huffman_coding_length', 'memorization_score'],
    num_rows: 2382328
})

In [15]:
deduped_12b = final_ds['memories_deduped_12b'].to_pandas()[datasets['memories_duped_12b.103000'].columns]

In [16]:
datasets['memories_deduped_12b.143000'] = deduped_12b

In [17]:
duped_12b = final_ds['memories_duped_12b'].to_pandas()[datasets['memories_duped_12b.103000'].columns]

In [18]:
datasets['memories_duped_12b.143000'] = duped_12b

In [19]:
from datasets import Dataset

In [20]:
for key, value in tqdm(datasets.items()):
    dataset = Dataset.from_pandas(value)
    dataset.push_to_hub("usvsnsp/semantic-filters-intermediate", split=key, max_shard_size="10GB")

  0%|          | 0/14 [00:00<?, ?it/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1511 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1997 [00:00<?, ?ba/s]

...

Creating parquet from Arrow format:   0%|          | 0/1872 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/2.97k [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/2383 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/3.15k [00:00<?, ?B/s]